# Batch ResNet Places365 untuk Dataset Skema 2

Notebook ini membaca `dataset/labels_skema2.xlsx`, mencocokkan kolom `File` dengan gambar di `dataset/images_skema2`, lalu menambahkan prediksi top-3 scene Places365 sebagai kolom `Scene1`, `Scene2`, `Scene3`, `Prob1`, `Prob2`, dan `Prob3`. Hasil akhirnya disimpan ke `dataset/sence-top3.csv`.

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image


def find_existing_path(*candidates: str) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f"Tidak ada path yang ditemukan dari kandidat: {candidates}")


DATASET_XLSX = find_existing_path("dataset/labels_skema2.xlsx", "../dataset/labels_skema2.xlsx")
IMAGE_DIR = find_existing_path("dataset/images_skema2", "../dataset/images_skema2")
MODEL_DIR = find_existing_path("models", "../models")
CATEGORIES_PATH = MODEL_DIR / "categories_places365.txt"
OUTPUT_CSV = DATASET_XLSX.parent / "sence-top3.csv"

ARCH = "resnet50" if (MODEL_DIR / "resnet50_places365.pth.tar").exists() else "resnet18"
MODEL_PATH = MODEL_DIR / f"{ARCH}_places365.pth.tar"
TOP_K = 3
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SCENE_COLUMNS = [f"Scene{i}" for i in range(1, TOP_K + 1)]
PROB_COLUMNS = [f"Prob{i}" for i in range(1, TOP_K + 1)]

print(f"Dataset: {DATASET_XLSX}")
print(f"Folder gambar: {IMAGE_DIR}")
print(f"Model: {MODEL_PATH}")
print(f"Output CSV: {OUTPUT_CSV}")
print(f"Arsitektur: {ARCH}")
print(f"Device: {DEVICE}")

In [ ]:
XLSX_NS = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
COLUMN_RE = re.compile(r"([A-Z]+)")
BUILTIN_DATE_FORMAT_IDS = {14, 15, 16, 17, 22, 27, 30, 36, 45, 46, 47, 50, 57}


def column_ref_to_index(cell_ref: str) -> int:
    match = COLUMN_RE.match(cell_ref)
    if not match:
        raise ValueError(f"Referensi cell tidak valid: {cell_ref}")

    index = 0
    for char in match.group(1):
        index = index * 26 + (ord(char) - ord("A") + 1)
    return index - 1


def excel_serial_to_datetime_text(value: str) -> str:
    try:
        total_seconds = round(float(value) * 86400)
    except (TypeError, ValueError):
        return value

    parsed = datetime(1899, 12, 30) + timedelta(seconds=total_seconds)
    return parsed.strftime("%Y-%m-%d %H:%M:%S")


def load_date_style_ids(zip_file: ZipFile) -> set[str]:
    if "xl/styles.xml" not in zip_file.namelist():
        return set()

    styles = ET.fromstring(zip_file.read("xl/styles.xml"))
    custom_formats = {}
    num_fmts = styles.find("a:numFmts", XLSX_NS)
    if num_fmts is not None:
        for num_fmt in num_fmts.findall("a:numFmt", XLSX_NS):
            custom_formats[int(num_fmt.attrib["numFmtId"])] = num_fmt.attrib.get("formatCode", "").lower()

    date_style_ids = set()
    cell_xfs = styles.find("a:cellXfs", XLSX_NS)
    if cell_xfs is None:
        return date_style_ids

    for style_index, xf in enumerate(cell_xfs.findall("a:xf", XLSX_NS)):
        num_fmt_id = int(xf.attrib.get("numFmtId", 0))
        format_code = custom_formats.get(num_fmt_id, "")
        is_builtin_date = num_fmt_id in BUILTIN_DATE_FORMAT_IDS
        is_custom_date = any(token in format_code for token in ["yy", "dd", "hh"])
        if is_builtin_date or is_custom_date:
            date_style_ids.add(str(style_index))

    return date_style_ids


def load_shared_strings(zip_file: ZipFile) -> list[str]:
    if "xl/sharedStrings.xml" not in zip_file.namelist():
        return []

    shared_root = ET.fromstring(zip_file.read("xl/sharedStrings.xml"))
    shared_strings = []
    for item in shared_root.findall("a:si", XLSX_NS):
        shared_strings.append("".join(text.text or "" for text in item.findall(".//a:t", XLSX_NS)))
    return shared_strings


def read_xlsx_first_sheet(path: Path) -> pd.DataFrame:
    with ZipFile(path) as zip_file:
        shared_strings = load_shared_strings(zip_file)
        date_style_ids = load_date_style_ids(zip_file)
        sheet = ET.fromstring(zip_file.read("xl/worksheets/sheet1.xml"))

        row_maps = []
        max_column_count = 0
        for row in sheet.findall(".//a:sheetData/a:row", XLSX_NS):
            row_values = {}
            for cell in row.findall("a:c", XLSX_NS):
                column_index = column_ref_to_index(cell.attrib["r"])
                max_column_count = max(max_column_count, column_index + 1)
                cell_type = cell.attrib.get("t")

                if cell_type == "inlineStr":
                    value = "".join(text.text or "" for text in cell.findall(".//a:t", XLSX_NS))
                else:
                    value_node = cell.find("a:v", XLSX_NS)
                    if value_node is None:
                        value = ""
                    elif cell_type == "s":
                        value = shared_strings[int(value_node.text)]
                    else:
                        value = value_node.text or ""
                        if cell.attrib.get("s") in date_style_ids:
                            value = excel_serial_to_datetime_text(value)

                row_values[column_index] = value
            row_maps.append(row_values)

        parsed_rows = [
            [row_values.get(index, "") for index in range(max_column_count)]
            for row_values in row_maps
        ]

    headers = parsed_rows[0]
    rows = parsed_rows[1:]
    while headers and headers[-1] == "":
        headers.pop()
        rows = [row[:-1] for row in rows]

    return pd.DataFrame(rows, columns=headers)


df = read_xlsx_first_sheet(DATASET_XLSX)
display(df.head())
print(f"Jumlah baris label: {len(df)}")
print("Kolom:", list(df.columns))

In [ ]:
if "File" not in df.columns:
    raise KeyError("Kolom 'File' tidak ditemukan di labels_skema2.xlsx")
if "Catatan" not in df.columns:
    raise KeyError("Kolom 'Catatan' tidak ditemukan di labels_skema2.xlsx")

file_series = df["File"].astype("string").fillna("").str.strip()
files_from_excel = [file_name for file_name in file_series.tolist() if file_name]
image_paths_by_name = {
    path.name: path
    for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
}

missing_images = sorted(set(files_from_excel) - set(image_paths_by_name))
extra_images = sorted(set(image_paths_by_name) - set(files_from_excel))

print(f"Baris dengan File terisi: {len(files_from_excel)}")
print(f"Gambar di folder images_skema2: {len(image_paths_by_name)}")
print(f"File dari Excel yang tidak ada di folder: {len(missing_images)}")
print(f"Gambar di folder yang tidak ada di Excel: {len(extra_images)}")

if missing_images:
    display(pd.DataFrame({"missing_images": missing_images}))
    raise FileNotFoundError("Ada nama file di kolom 'File' yang tidak ditemukan di folder images_skema2.")

if extra_images:
    display(pd.DataFrame({"extra_images": extra_images}))

ordered_image_paths = [image_paths_by_name[file_name] for file_name in files_from_excel]

In [ ]:
def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        padding=dilation,
        groups=groups,
        bias=False,
        dilation=dilation,
    )


def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1, base_width=64, dilation=1, norm_layer=None):
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        if groups != 1 or base_width != 64:
            raise ValueError("BasicBlock hanya mendukung groups=1 dan base_width=64")
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 belum didukung untuk BasicBlock")

        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1, base_width=64, dilation=1, norm_layer=None):
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        width = int(planes * (base_width / 64.0)) * groups

        self.conv1 = conv1x1(inplanes, width)
        self.bn1 = norm_layer(width)
        self.conv2 = conv3x3(width, width, stride, groups, dilation)
        self.bn2 = norm_layer(width)
        self.conv3 = conv1x1(width, planes * self.expansion)
        self.bn3 = norm_layer(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=365, groups=1, width_per_group=64, replace_stride_with_dilation=None, norm_layer=None):
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self._norm_layer = norm_layer
        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        self.groups = groups
        self.base_width = width_per_group

        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = [
            block(
                self.inplanes,
                planes,
                stride,
                downsample,
                self.groups,
                self.base_width,
                previous_dilation,
                norm_layer,
            )
        ]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups, base_width=self.base_width, dilation=self.dilation, norm_layer=norm_layer))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def build_resnet_places365(arch: str) -> nn.Module:
    if arch == "resnet18":
        return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=365)
    if arch == "resnet50":
        return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=365)
    raise ValueError("ARCH harus 'resnet18' atau 'resnet50'")

In [ ]:
def load_categories(path: Path) -> list[str]:
    categories = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        raw_name = line.split()[0]
        scene_name = raw_name.split("/")[-1].replace("_", " ")
        categories.append(scene_name)
    return categories


def clean_state_dict(checkpoint):
    state_dict = checkpoint.get("state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    cleaned = {}
    for key, value in state_dict.items():
        cleaned[key.replace("module.", "")] = value
    return cleaned


if not CATEGORIES_PATH.exists():
    raise FileNotFoundError(f"File kategori tidak ditemukan: {CATEGORIES_PATH}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"File model tidak ditemukan: {MODEL_PATH}")

categories = load_categories(CATEGORIES_PATH)
if len(categories) != 365:
    raise ValueError(f"Jumlah kategori harus 365, tetapi terbaca {len(categories)}")

model = build_resnet_places365(ARCH)
try:
    checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(MODEL_PATH, map_location="cpu")

model.load_state_dict(clean_state_dict(checkpoint), strict=True)
model = model.to(DEVICE)
model.eval()

print(f"Model {ARCH} Places365 siap dipakai.")
print(f"Jumlah kategori scene: {len(categories)}")

In [ ]:
BICUBIC = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def resize_for_places365(image: Image.Image, size: int = 256) -> Image.Image:
    return image.resize((size, size), BICUBIC)


def center_crop(image: Image.Image, size: int = 224) -> Image.Image:
    width, height = image.size
    left = max((width - size) // 2, 0)
    top = max((height - size) // 2, 0)
    return image.crop((left, top, left + size, top + size))


def preprocess_image(image_path: Path) -> torch.Tensor:
    image = Image.open(image_path).convert("RGB")
    image = resize_for_places365(image, 256)
    image = center_crop(image, 224)

    array = np.asarray(image).astype("float32") / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1)
    return (tensor - IMAGENET_MEAN) / IMAGENET_STD


@torch.inference_mode()
def predict_scenes_batch(image_paths: list[Path], top_k: int = TOP_K, batch_size: int = BATCH_SIZE) -> dict[str, list[tuple[str, float]]]:
    predictions = {}
    total = len(image_paths)

    for start in range(0, total, batch_size):
        batch_paths = image_paths[start : start + batch_size]
        batch_tensors = []
        valid_paths = []

        for image_path in batch_paths:
            try:
                batch_tensors.append(preprocess_image(image_path))
                valid_paths.append(image_path)
            except Exception as exc:
                print(f"Gagal memproses {image_path.name}: {exc}")

        if not batch_tensors:
            continue

        inputs = torch.stack(batch_tensors).to(DEVICE)
        logits = model(inputs)
        probabilities = torch.softmax(logits, dim=1)
        top_probabilities, top_indices = probabilities.topk(top_k, dim=1)

        for image_path, indices, probabilities_row in zip(valid_paths, top_indices.cpu().tolist(), top_probabilities.cpu().tolist()):
            predictions[image_path.name] = [
                (categories[index], float(probability))
                for index, probability in zip(indices, probabilities_row)
            ]

        done = min(start + len(batch_paths), total)
        print(f"Progress prediksi: {done}/{total} gambar", end="\r")

    print(f"Progress prediksi: {total}/{total} gambar")
    return predictions

In [ ]:
scene_predictions = predict_scenes_batch(ordered_image_paths)

result_df = df.copy()
for column in SCENE_COLUMNS + PROB_COLUMNS:
    result_df[column] = ""

for row_index, file_name in file_series.items():
    if not file_name:
        continue

    for scene_index, (scene_name, probability) in enumerate(scene_predictions.get(file_name, []), start=1):
        result_df.at[row_index, f"Scene{scene_index}"] = scene_name
        result_df.at[row_index, f"Prob{scene_index}"] = round(probability, 6)

base_columns = list(df.columns)
insert_position = base_columns.index("Catatan") + 1
ordered_columns = base_columns[:insert_position] + SCENE_COLUMNS + PROB_COLUMNS + base_columns[insert_position:]
result_df = result_df[ordered_columns]

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"CSV berhasil disimpan: {OUTPUT_CSV}")
print(f"Jumlah baris output: {len(result_df)}")
display(result_df.head())